# meshio Round-trip: FieldML -> VTU -> FieldML

`pyfieldml` includes a first-class two-way bridge to `meshio`, which unlocks every I/O format meshio supports (VTU, XDMF, Gmsh, Abaqus, Ansys, Nastran, ...). This notebook round-trips a dataset through `.vtu`, **renders before/after visually**, and verifies that node coordinates survive unchanged.

In [ ]:
import tempfile
from pathlib import Path

import meshio
import numpy as np

import pyfieldml as fml
from pyfieldml import datasets

## Load a bundled dataset

In [ ]:
doc = datasets.load_femur_bodyparts3d()
coords_before = doc.evaluators["coordinates"].as_ndarray()
print("nodes:", coords_before.shape)

## Convert to meshio and write as VTU

In [ ]:
m = doc.to_meshio()
print("meshio cells:", m.cells[0].type, "x", len(m.cells[0].data))

out_dir = Path(tempfile.mkdtemp(prefix="fieldml_rt_"))
vtu_path = out_dir / "femur.vtu"
meshio.write(vtu_path, m)
print("wrote", vtu_path, f"({vtu_path.stat().st_size} bytes)")

## Read the VTU back with meshio and re-import into pyfieldml

In [ ]:
m2 = meshio.read(vtu_path)
doc2 = fml.Document.from_meshio(m2, name="roundtrip")
coords_after = doc2.evaluators["coordinates"].as_ndarray()
print("nodes after round-trip:", coords_after.shape)

## Side-by-side rendering

Before / after should be visually indistinguishable - which is the whole point of lossless coordinate round-trip.

In [ ]:
import pyvista as pv

from pyfieldml.interop.pyvista import to_pyvista

pv.OFF_SCREEN = True
pv.set_jupyter_backend("static")

g1 = to_pyvista(doc)
g2 = to_pyvista(doc2)
p = pv.Plotter(off_screen=True, window_size=(1000, 450), shape=(1, 2))
p.subplot(0, 0)
p.add_mesh(g1, color="ivory", show_edges=False)
p.add_text("source FieldML", font_size=10)
p.view_isometric()
p.subplot(0, 1)
p.add_mesh(g2, color="peachpuff", show_edges=False)
p.add_text("round-tripped via VTU", font_size=10)
p.view_isometric()
p.show(screenshot="/tmp/roundtrip.png")

## Quantitative coordinate diff

Sort-and-compare because meshio/VTU do not preserve the original node ordering in general.

In [ ]:
import matplotlib.pyplot as plt

a = np.sort(coords_before, axis=0)
b = np.sort(coords_after, axis=0)
delta = np.abs(a - b)

fig, ax = plt.subplots(figsize=(7, 3))
ax.semilogy(delta[:, 0] + 1e-20, label="|dx|")
ax.semilogy(delta[:, 1] + 1e-20, label="|dy|")
ax.semilogy(delta[:, 2] + 1e-20, label="|dz|")
ax.set_xlabel("sorted node index")
ax.set_ylabel("|coord_after - coord_before|")
ax.set_title("Round-trip coordinate residuals (should be at float noise)")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
fig.tight_layout()

## Verify node equality

In [ ]:
assert coords_before.shape == coords_after.shape, (
    f"shape mismatch: {coords_before.shape} vs {coords_after.shape}"
)
np.testing.assert_allclose(a, b, atol=1e-12)
print("OK: node coordinates survive the FieldML -> VTU -> FieldML round-trip.")

### What's lost, what's kept

**Kept**: node coordinates, element connectivity, per-node ParameterEvaluators (passed through `meshio.Mesh.point_data`).

**Lost**: the evaluator graph (basis external references, scale fields, derivative DOFs). meshio is a mesh-I/O layer and has no representation for FieldML's evaluator hierarchy, so exporting to VTU is lossy. Keep `.fieldml` for archival; use VTU/XDMF for visualization and FEM handoff.